<a href="https://colab.research.google.com/github/KhanakKRDS/AI-text-summary/blob/main/fine_tune_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone git@git.drupal.org:project/<AI-text-summary>.git


/bin/bash: line 1: AI-text-summary: No such file or directory


In [ ]:
!git remote add origin git@git.drupal.org:project/<AI-text-summary>.git


/bin/bash: line 1: AI-text-summary: No such file or directory


In [ ]:
import nltk
from datasets import load_dataset
from transformers import(
    BartTokenizer,
    BartForConditionalGeneration,# pre-trained model
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
    DataCollatorForSeq2Seq,
    )
from sklearn.model_selection import train_test_split
##from transformers import TrainingArguments
##from transformers import EarlyStoppingCallback
##from transformers import DataCollatorForSeq2Seq
##from transformers import Seq2SeqTrainer

#load and subset he dataset
dataset = load_dataset ("d0rj/wikisum") #wiki_sum for fine-tuning the model
subset = dataset["train"].select(range(500)) #select 10000 examples instead of full dataset(reduce memory usage)
train_idx, val_idx = train_test_split(range(len(subset)), test_size = 0.5, random_state = 42)
subset_train = subset.select(train_idx)
subset_val = subset.select(val_idx)

#load okenizer and model
tokenizer = BartTokenizer.from_pretrained ("facebook/bart-large-cnn") #BART tokenizer
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn") #BART model

#tokenization (dynamically adjusts padding and trunction)
def preprocess(examples):
    #truncation - limiting the number of digits right of the decimal point
    model_inputs = tokenizer(examples["article"], max_length = 512, truncation=True, padding = "longest")
    labels = tokenizer(examples["summary"], max_length = 256, truncation=True, padding = "longest")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

#tokeniztion while removing large articles for efficiency
tokenized_train = subset_train.map(preprocess, batched=True, remove_columns = ["article", "summary"])
tokenized_val = subset_val.map(preprocess, batched=True, remove_columns = ["article", "summary"])
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)#datacollator for seq2seq models

#define training prameters with optimizations
training_args = Seq2SeqTrainingArguments(
    output_dir = "./checkpoints", #saves model checkpoints and final model (place where this script is saved)
    eval_strategy = "epoch", #evaluate after each epoch
    save_strategy = "epoch",
    learning_rate = 3e-5, #fine-tuning learning rates
    per_device_train_batch_size = 2,#small batch size to prevent memory overload
    gradient_accumulation_steps = 16, #accumulate gradients for larger effective batches
    per_device_eval_batch_size = 2, #evaluation batch size
    weight_decay = 0.01, #regularization to prevent overfitting
    save_total_limit = 2, #limits number of saved checkpoints
    num_train_epochs = 2, #number of training epochs
    logging_dir = "./logs", #saves logs for monitoring training(via TensorBoard)
    logging_steps = 50, #log training progress every 50 steps
    report_to = ["tensorboard"], #enables tensorboard for tracking performances
    metric_for_best_model = "eval_loss",
    load_best_model_at_end = True,
    greater_is_better = False, #lower eval_loss is better
    predict_with_generate = True,
    )

#train model on selected dataset(define trainer with early stopping)
trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,  #using 20k samples for training
    eval_dataset = tokenized_val, #using 2k samples for validation
    data_collator = data_collator,
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 2)], #stops training if validation loss doesn't improve
    )

#train the model with the improved settings
trainer.train() #train the model

#save the fine-tuned model
model.save_pretrained("My-tune-model")
tokenizer.save_pretrained("My-tune-model")

print("model fine-tuned and saved to My-tune-model")



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,No log,2.854094
2,No log,1.996574


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


model fine-tuned and saved to My-tune-model


In [ ]:
pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [ ]:
pip uninstall fsspec

Found existing installation: fsspec 2025.5.1
Uninstalling fsspec-2025.5.1:
  Would remove:
    /usr/local/lib/python3.11/dist-packages/fsspec-2025.5.1.dist-info/*
    /usr/local/lib/python3.11/dist-packages/fsspec/*
Proceed (Y/n)? y
  Successfully uninstalled fsspec-2025.5.1


In [ ]:
pip install fsspec==2025.3.2

  Using cached fsspec-2025.3.2-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2025.3.2-py3-none-any.whl (194 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.6.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; p

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!rm -rf ~/.cache/huggingface/datasets/d0rj__wikisum